# FEATURE ENGINEERING

## Import Necessary Libraries

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path


from sklearn.preprocessing import LabelEncoder
from sklearn.feature_selection import VarianceThreshold

import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

## 1. Loading Cleaned Dataset

The dataset used in this notebook is the output from the EDA phase. At this stage, the data has already undergone:

- Missing value treatment
- Missingness indicator creation
- Initial data quality checks
- Constant feature removal

The focus now shifts to feature engineering, where additional predictive features will be created and redundant variables will be removed before modelling.

In [2]:
OUTPUT_DIR = Path("./output")

train_fe = pd.read_parquet(OUTPUT_DIR / "train_clean_eda.parquet")
test_fe = pd.read_parquet(OUTPUT_DIR / "test_clean_eda.parquet")

print("Train shape:", train_fe.shape)
print("Test shape:", test_fe.shape)

Train shape: (307511, 678)
Test shape: (48744, 677)


## 2. Separate Target and IDs


In [3]:

# Separate target and ID columns


# Save the target variable
TARGET = train_fe["TARGET"]

# Save applicant IDs
train_ids = train_fe["SK_ID_CURR"]
test_ids = test_fe["SK_ID_CURR"]

# Remove TARGET from training features
train_fe = train_fe.drop(columns=["TARGET"])

# Remove ID columns from both datasets
train_fe = train_fe.drop(columns=["SK_ID_CURR"])
test_fe = test_fe.drop(columns=["SK_ID_CURR"])

print("Feature engineering train shape:", train_fe.shape)
print("Feature engineering test shape:", test_fe.shape)

Feature engineering train shape: (307511, 676)
Feature engineering test shape: (48744, 676)


I separated the target and ID columns because they should not be treated as input features during feature engineering.

## 3. Identify Feature Types

In [4]:
# Identify numeric and categorical features

numeric_cols = train_fe.select_dtypes(include="number").columns.tolist()
categorical_cols = train_fe.select_dtypes(include=["object"]).columns.tolist()

print(f"Numeric features: {len(numeric_cols)}")
print(f"Categorical features: {len(categorical_cols)}")

Numeric features: 660
Categorical features: 16


## 4. Create New Features

 The original Home Credit dataset contains a lot of useful information, but many variables become more informative when they are combined. Instead of relying only on the raw values, I will create features that will better describe a customer's financial situation, employment history, age, and credit behaviour

#### Step 1: Copy datasets

This prevents modifying the original data

In [5]:
# Create working copies for feature engineering
train_eng = train_fe.copy()
test_eng = test_fe.copy()

#### Step 2: Helper Function
This makes sure I don't divide by zero.

In [6]:
def safe_divide(a, b):
    return a / (b.replace(0, np.nan) + 1e-8)

#### Step 3: Financial Ratio Features

In [7]:
datasets = [train_eng, test_eng]

for df in datasets:

    # Credit relative to annual income
    df["CREDIT_INCOME_RATIO"] = safe_divide(
        df["AMT_CREDIT"],
        df["AMT_INCOME_TOTAL"]
    )

    # Loan repayment burden
    df["ANNUITY_INCOME_RATIO"] = safe_divide(
        df["AMT_ANNUITY"],
        df["AMT_INCOME_TOTAL"]
    )

    # Loan compared to goods value
    df["CREDIT_GOODS_RATIO"] = safe_divide(
        df["AMT_CREDIT"],
        df["AMT_GOODS_PRICE"]
    )

    # Goods affordability
    df["GOODS_INCOME_RATIO"] = safe_divide(
        df["AMT_GOODS_PRICE"],
        df["AMT_INCOME_TOTAL"]
    )

    # Loan repayment relative to loan size
    df["ANNUITY_CREDIT_RATIO"] = safe_divide(
        df["AMT_ANNUITY"],
        df["AMT_CREDIT"]
    )

The code creates five financial ratio features that describe the relationship between an applicant's income, loan amount, repayments, and the value of the financed goods. These relationships provide more meaningful information than the original financial variables alone. Rather than relying only on absolute values, the engineered features capture affordability, repayment burden, and borrowing behaviour, helping the machine learning model better distinguish between applicants who are more or less likely to default on their loans.

#### Step 4 : Family Features

In [8]:
for df in datasets:

    df["INCOME_PER_PERSON"] = safe_divide(
        df["AMT_INCOME_TOTAL"],
        df["CNT_FAM_MEMBERS"]
    )

    df["INCOME_PER_CHILD"] = safe_divide(
        df["AMT_INCOME_TOTAL"],
        df["CNT_CHILDREN"] + 1
    )

    df["CREDIT_PER_PERSON"] = safe_divide(
        df["AMT_CREDIT"],
        df["CNT_FAM_MEMBERS"]
    )

    df["CHILDREN_RATIO"] = safe_divide(
        df["CNT_CHILDREN"],
        df["CNT_FAM_MEMBERS"]
    )

This block creates household affordability features. Instead of considering only the applicant's total income or loan amount, these features take household size into account. This provides a more realistic picture of the applicant's financial capacity because income and debt are often shared across family members.

It creates four household affordability features that combine income, loan amount, and family composition. The original variables describe these factors separately, but the engineered features capture how they interact. By accounting for household size and the number of dependants, these features provide a more realistic measure of an applicant's financial capacity and obligations. This additional information can help the machine learning model better identify differences in repayment ability and improve the prediction of loan default.

#### Step: 5 Age Features

Convert negative days into years.

In [9]:
for df in datasets:

    df["AGE_YEARS"] = -df["DAYS_BIRTH"] / 365

    df["EMPLOYMENT_YEARS"] = -df["DAYS_EMPLOYED"] / 365

    df["REGISTRATION_YEARS"] = -df["DAYS_REGISTRATION"] / 365

    df["ID_PUBLISH_YEARS"] = -df["DAYS_ID_PUBLISH"] / 365

It creates time-based features by converting variables measured in negative days into positive years. In the Home Credit dataset, these variables are stored as the number of days before the loan application date, which makes them difficult to interpret. Converting them into years makes the data more understandable while preserving the same underlying information.

#### Step 6: Employment Features

In [10]:
for df in datasets:

    df["EMPLOYMENT_AGE_RATIO"] = safe_divide(
        df["EMPLOYMENT_YEARS"],
        df["AGE_YEARS"]
    )

    df["REGISTRATION_AGE_RATIO"] = safe_divide(
        df["REGISTRATION_YEARS"],
        df["AGE_YEARS"]
    )

The code creates two life-stage ratio features by comparing employment duration and registration duration to the applicant's age. These ratios provide context that the original variables cannot capture on their own. Instead of simply measuring how long an applicant has worked or been registered, the features show how significant those periods are relative to the applicant's lifetime. This helps the model better distinguish between applicants with different levels of employment and residential stability, which are factors that can influence the likelihood of loan repayment.

#### Step 7: External Credit Score Features

These are among the strongest predictors in the Home Credit dataset.

In [11]:
ext_cols = [
    c for c in [
        "EXT_SOURCE_1",
        "EXT_SOURCE_2",
        "EXT_SOURCE_3"
    ]
    if c in train_eng.columns
]

for df in datasets:

    df["EXT_SOURCE_MEAN"] = df[ext_cols].mean(axis=1)

    df["EXT_SOURCE_MAX"] = df[ext_cols].max(axis=1)

    df["EXT_SOURCE_MIN"] = df[ext_cols].min(axis=1)

    df["EXT_SOURCE_STD"] = df[ext_cols].std(axis=1)

    df["EXT_SOURCE_SUM"] = df[ext_cols].sum(axis=1)

The code creates summary statistics from the three external credit score variables: EXT_SOURCE_1, EXT_SOURCE_2, and EXT_SOURCE_3. These variables are among the most important predictors in the Home Credit dataset because they represent credit scores obtained from external sources. During my exploratory data analysis (EDA),I  found that these variables had some of the strongest correlations with the target (TARGET), making them highly valuable for predicting loan default.

Rather than using each score independently, this code combines them into new features that summarize an applicant's overall external credit profile.

#### Step 8: Loan Duration Estimate

A useful approximation of how long the loan would take to repay

In [12]:
for df in datasets:

    df["LOAN_TERM"] = safe_divide(
        df["AMT_CREDIT"],
        df["AMT_ANNUITY"]
    )

The code creates one financial feature that estimates the loan repayment period by combining the total loan amount and the annual annuity payment. While the original variables describe the loan and its repayments separately, LOAN_TERM captures their relationship in a single, more informative measure. This engineered feature helps the machine learning model better understand the repayment structure of each loan and may improve its ability to distinguish between applicants with different borrowing characteristics and levels of credit risk.

#### Step 9: Social Risk Features

In [13]:
for df in datasets:

    df["TOTAL_SOCIAL_OBS"] = (
        df["OBS_30_CNT_SOCIAL_CIRCLE"] +
        df["OBS_60_CNT_SOCIAL_CIRCLE"]
    )

    df["TOTAL_SOCIAL_DEF"] = (
        df["DEF_30_CNT_SOCIAL_CIRCLE"] +
        df["DEF_60_CNT_SOCIAL_CIRCLE"]
    )

The code creates two summary features that measure the overall level of payment difficulties and loan defaults within an applicant's social circle. The original variables capture these events over separate 30-day and 60-day periods, but combining them provides a more comprehensive view of the applicant's social environment. These engineered features help the model use the information more efficiently by summarizing related variables into single indicators, potentially improving its ability to identify patterns associated with credit risk.

#### Step 10: Flag Features

These combine related binary indicators into more informative counts.

In [14]:
for df in datasets:

    df["TOTAL_CONTACT_FLAGS"] = (
        df["FLAG_MOBIL"] +
        df["FLAG_EMP_PHONE"] +
        df["FLAG_WORK_PHONE"] +
        df["FLAG_CONT_MOBILE"] +
        df["FLAG_PHONE"] +
        df["FLAG_EMAIL"]
    )

This block creates one summary feature that represents the total number of contact methods provided by an applicant. The original dataset stores each contact method in a separate binary variable, but these variables all describe the same underlying characteristic-the applicant's availability through different communication channels. By combining them into TOTAL_CONTACT_FLAGS, the feature engineering process creates a more compact and informative measure of applicant accessibility. This simplifies the dataset while preserving valuable information that may help the machine learning model distinguish between applicants with different levels of stability and credit risk.

#### Step 11 :Check the New Features

In [15]:
# Identify newly created features
new_features = sorted(
    list(set(train_eng.columns) - set(train_fe.columns))
)

print(f"New features created: {len(new_features)}")

new_features

New features created: 24


['AGE_YEARS',
 'ANNUITY_CREDIT_RATIO',
 'ANNUITY_INCOME_RATIO',
 'CHILDREN_RATIO',
 'CREDIT_GOODS_RATIO',
 'CREDIT_INCOME_RATIO',
 'CREDIT_PER_PERSON',
 'EMPLOYMENT_AGE_RATIO',
 'EMPLOYMENT_YEARS',
 'EXT_SOURCE_MAX',
 'EXT_SOURCE_MEAN',
 'EXT_SOURCE_MIN',
 'EXT_SOURCE_STD',
 'EXT_SOURCE_SUM',
 'GOODS_INCOME_RATIO',
 'ID_PUBLISH_YEARS',
 'INCOME_PER_CHILD',
 'INCOME_PER_PERSON',
 'LOAN_TERM',
 'REGISTRATION_AGE_RATIO',
 'REGISTRATION_YEARS',
 'TOTAL_CONTACT_FLAGS',
 'TOTAL_SOCIAL_DEF',
 'TOTAL_SOCIAL_OBS']

#### Step 12: Check that they were created correctly

In [16]:
print(f"Number of new features: {len(new_features)}")

train_eng[new_features].head()

Number of new features: 24


,AGE_YEARS,ANNUITY_CREDIT_RATIO,ANNUITY_INCOME_RATIO,CHILDREN_RATIO,CREDIT_GOODS_RATIO,CREDIT_INCOME_RATIO,CREDIT_PER_PERSON,EMPLOYMENT_AGE_RATIO,EMPLOYMENT_YEARS,EXT_SOURCE_MAX,EXT_SOURCE_MEAN,EXT_SOURCE_MIN,EXT_SOURCE_STD,EXT_SOURCE_SUM,GOODS_INCOME_RATIO,ID_PUBLISH_YEARS,INCOME_PER_CHILD,INCOME_PER_PERSON,LOAN_TERM,REGISTRATION_AGE_RATIO,REGISTRATION_YEARS,TOTAL_CONTACT_FLAGS,TOTAL_SOCIAL_DEF,TOTAL_SOCIAL_OBS
0,25.920548,0.060749,0.121978,0.0,1.158397,2.007889,406597.495934,0.067329,1.745205,0.262949,0.161787,0.083037,0.092026,0.485361,1.733333,5.808219,202499.997975,202499.997975,16.461104,0.385583,9.994521,4.0,4.0,4.0
1,45.931507,0.027598,0.132217,0.0,1.145199,4.790750,646751.246766,0.070862,3.254795,0.622246,0.489596,0.311267,0.160443,1.468789,4.183333,0.797260,269999.997300,134999.999325,36.234085,0.070743,3.249315,4.0,0.0,2.0
2,52.180822,0.050000,0.100000,0.0,1.000000,2.000000,134999.998650,0.011814,0.616438,0.729567,0.597159,0.505998,0.117353,1.791477,2.000000,6.934247,67499.999325,67499.999325,20.000000,0.223669,11.671233,5.0,0.0,0.0
3,52.068493,0.094941,0.219900,0.0,1.052803,2.316167,156341.249218,0.159905,8.326027,0.650442,0.563905,0.505998,0.076359,1.691716,2.200000,6.676712,134999.998650,67499.999663,10.532818,0.517390,26.939726,3.0,0.0,4.0
4,54.608219,0.042623,0.179963,0.0,1.000000,4.222222,512999.994870,0.152418,8.323288,0.535276,0.454671,0.322738,0.115191,1.364012,4.222222,9.473973,121499.998785,121499.998785,23.461618,0.216285,11.810959,3.0,0.0,0.0


#### Step 13: Check for Missing Values

In [17]:
missing_report = (
    train_eng[new_features]
    .isna()
    .sum()
    .sort_values(ascending=False)
)

display(missing_report[missing_report > 0])

Series([], dtype: int64)

### Step 14: Check for Infinite Values

Ratio features sometimes produce infinity.

In [18]:

inf_report = (
    train_eng[new_features]
    .replace([np.inf, -np.inf], np.nan)
    .isna()
    .sum()
    .sort_values(ascending=False)
)

display(inf_report[inf_report > 0])

Series([], dtype: int64)

#### Step 15: Check Distribution

In [19]:
train_eng[new_features].describe().T

,count,mean,std,min,25%,50%,75%,max
AGE_YEARS,307511.0,43.936973,11.956133,2.051781e+01,34.008219,43.150685,53.923288,6.912055e+01
ANNUITY_CREDIT_RATIO,307511.0,0.053695,0.022482,1.678969e-02,0.036900,0.050000,0.064043,1.581143e-01
ANNUITY_INCOME_RATIO,307511.0,0.180929,0.094573,2.238846e-04,0.114782,0.162833,0.229067,1.875965e+00
CHILDREN_RATIO,307511.0,0.125502,0.199578,0.000000e+00,0.000000,0.000000,0.333333,9.500000e-01
CREDIT_GOODS_RATIO,307511.0,1.122542,0.125542,1.500000e-01,1.000000,1.118800,1.198000,6.000000e+00
CREDIT_INCOME_RATIO,307511.0,3.957570,2.689728,4.807615e-03,2.018667,3.265067,5.159880,8.473684e+01
CREDIT_PER_PERSON,307511.0,323949.527364,259086.212903,6.750000e+03,135533.249322,255050.998725,437369.998542,4.031032e+06
EMPLOYMENT_AGE_RATIO,307511.0,0.142371,0.124883,-0.000000e+00,0.066852,0.091037,0.191054,7.288115e-01
EMPLOYMENT_YEARS,307511.0,6.168784,5.852585,-0.000000e+00,2.556164,4.515068,7.561644,4.907397e+01
EXT_SOURCE_MAX,307511.0,0.642007,0.109909,3.297728e-02,0.540662,0.648336,0.725276,9.626928e-01


The engineered features were validated by checking for missing values, infinite values, and descriptive statistics. The summary statistics showed that all newly created variables had reasonable ranges and distributions, confirming that the feature engineering process was successfully implemented.

#### Step 16: Check Correlation with TARGET

In [20]:
train_corr = train_eng.copy()
train_corr["TARGET"] = TARGET
new_feature_corr = (
    train_corr[new_features + ["TARGET"]]
    .corr(numeric_only=True)["TARGET"]
    .drop("TARGET")
    .sort_values()
)

print("Top Positive Correlations")
display(new_feature_corr.tail(10))

print("Top Negative Correlations")
display(new_feature_corr.head(10))

Top Positive Correlations


CREDIT_INCOME_RATIO    -0.007727
INCOME_PER_PERSON      -0.006573
TOTAL_SOCIAL_OBS        0.009396
ANNUITY_CREDIT_RATIO    0.012698
ANNUITY_INCOME_RATIO    0.014268
TOTAL_CONTACT_FLAGS     0.020774
CHILDREN_RATIO          0.021224
TOTAL_SOCIAL_DEF        0.033111
CREDIT_GOODS_RATIO      0.068474
EXT_SOURCE_STD          0.078034
Name: TARGET, dtype: float64

Top Negative Correlations


EXT_SOURCE_SUM         -0.220840
EXT_SOURCE_MEAN        -0.220840
EXT_SOURCE_MIN         -0.192750
EXT_SOURCE_MAX         -0.174193
AGE_YEARS              -0.078239
EMPLOYMENT_YEARS       -0.063368
ID_PUBLISH_YEARS       -0.051457
EMPLOYMENT_AGE_RATIO   -0.049603
REGISTRATION_YEARS     -0.041975
LOAN_TERM              -0.032101
Name: TARGET, dtype: float64

##### Top Negative Correlations (Lower Risk of Default)
| Feature | Correlation | Interpretation |
|---------|------------:|---------------|
| **EXT_SOURCE_SUM** | -0.221 | Applicants with a higher combined external credit score were less likely to default. This was the strongest engineered feature. |
| **EXT_SOURCE_MEAN** | -0.221 | A higher average external credit score was linked to a lower chance of default. |
| **EXT_SOURCE_MIN** | -0.193 | Applicants with higher minimum external credit scores were less likely to default. |
| **EXT_SOURCE_MAX** | -0.174 | Applicants with higher maximum external credit scores also had a lower risk of default. |
| **AGE_YEARS** | -0.078 | Older applicants were slightly less likely to default on their loans. |
| **EMPLOYMENT_YEARS** | -0.063 | Applicants who had worked for more years tended to have a lower risk of default. |
| **ID_PUBLISH_YEARS** | -0.051 | Applicants whose identification records had been unchanged for longer showed a slightly lower risk of default. |
| **EMPLOYMENT_AGE_RATIO** | -0.050 | Applicants who had spent a larger part of their lives working were slightly less likely to default. |
| **REGISTRATION_YEARS** | -0.042 | Applicants who had been registered for longer showed a slightly lower risk of default. |
| **LOAN_TERM** | -0.032 | Longer loan repayment periods had a very weak relationship with lower default risk. |


##### Top Positive Correlations (Higher Risk of Default)
| Feature | Correlation | Interpretation |
|---------|------------:|---------------|
| **EXT_SOURCE_STD** | 0.078 | Applicants whose external credit scores varied more had a slightly higher risk of default. |
| **CREDIT_GOODS_RATIO** | 0.068 | Applicants borrowing more compared to the value of the goods they were buying were slightly more likely to default. |
| **TOTAL_SOCIAL_DEF** | 0.033 | Applicants with more defaults in their social circle had a slightly higher risk of default. |
| **CHILDREN_RATIO** | 0.021 | A higher number of children compared to family size showed a very weak relationship with default. |
| **TOTAL_CONTACT_FLAGS** | 0.021 | Applicants with more contact flags had a very small increase in default risk. |
| **ANNUITY_INCOME_RATIO** | 0.014 | Applicants spending a larger share of their income on loan repayments had a very weak increase in default risk. |
| **ANNUITY_CREDIT_RATIO** | 0.013 | Higher annual repayments compared to the loan amount showed a very weak positive relationship with default. |
| **TOTAL_SOCIAL_OBS** | 0.009 | This feature had almost no relationship with default. |
| **INCOME_PER_PERSON** | -0.007 | This feature showed almost no relationship with default. |
| **CREDIT_INCOME_RATIO** | -0.008 | This feature also showed almost no relationship with default. |

**Therefore,**
The engineered features improved the information available for predicting loan default. The strongest new features were the combined external credit score variables (EXT_SOURCE_SUM, EXT_SOURCE_MEAN, EXT_SOURCE_MIN, and EXT_SOURCE_MAX), which had stronger relationships with the target than the original external credit score features. This shows that combining related variables made them more useful for prediction. Features related to age, employment history, and customer registration also helped identify applicants with a lower risk of default. Although some engineered features had very weak correlations with the target, they can still be useful because tree-based machine learning models such as Random Forest, XGBoost, and LightGBM are able to learn complex, non-linear relationships that simple correlation cannot capture.

#### Final Quality Check

In [22]:
# Check for missing values
assert train_eng.isna().sum().sum() == 0
assert test_eng.isna().sum().sum() == 0

# Check for infinite values
assert np.isinf(train_eng.select_dtypes(include=np.number)).sum().sum() == 0
assert np.isinf(test_eng.select_dtypes(include=np.number)).sum().sum() == 0


print(" Feature engineering completed successfully.")

 Feature engineering completed successfully.


### Add back TARGET and IDs

In [23]:
# Restore TARGET and IDs

train_final = train_eng.copy()
test_final = test_eng.copy()

train_final["TARGET"] = TARGET

train_final["SK_ID_CURR"] = train_ids
test_final["SK_ID_CURR"] = test_ids

In [24]:
print(train_final.shape)
print(test_final.shape)

train_final.head()

(307511, 702)
(48744, 701)


,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,OWN_CAR_AGE,FLAG_MOBIL,FLAG_EMP_PHONE,FLAG_WORK_PHONE,FLAG_CONT_MOBILE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,REGION_RATING_CLIENT,REGION_RATING_CLIENT_W_CITY,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,REG_REGION_NOT_LIVE_REGION,REG_REGION_NOT_WORK_REGION,LIVE_REGION_NOT_WORK_REGION,REG_CITY_NOT_LIVE_CITY,REG_CITY_NOT_WORK_CITY,LIVE_CITY_NOT_WORK_CITY,ORGANIZATION_TYPE,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,APARTMENTS_AVG,BASEMENTAREA_AVG,YEARS_BEGINEXPLUATATION_AVG,YEARS_BUILD_AVG,COMMONAREA_AVG,ELEVATORS_AVG,ENTRANCES_AVG,FLOORSMAX_AVG,FLOORSMIN_AVG,LANDAREA_AVG,LIVINGAPARTMENTS_AVG,LIVINGAREA_AVG,NONLIVINGAPARTMENTS_AVG,NONLIVINGAREA_AVG,APARTMENTS_MODE,BASEMENTAREA_MODE,YEARS_BEGINEXPLUATATION_MODE,YEARS_BUILD_MODE,COMMONAREA_MODE,ELEVATORS_MODE,ENTRANCES_MODE,FLOORSMAX_MODE,FLOORSMIN_MODE,LANDAREA_MODE,LIVINGAPARTMENTS_MODE,LIVINGAREA_MODE,NONLIVINGAPARTMENTS_MODE,NONLIVINGAREA_MODE,APARTMENTS_MEDI,BASEMENTAREA_MEDI,YEARS_BEGINEXPLUATATION_MEDI,YEARS_BUILD_MEDI,COMMONAREA_MEDI,ELEVATORS_MEDI,ENTRANCES_MEDI,FLOORSMAX_MEDI,FLOORSMIN_MEDI,LANDAREA_MEDI,LIVINGAPARTMENTS_MEDI,LIVINGAREA_MEDI,NONLIVINGAPARTMENTS_MEDI,NONLIVINGAREA_MEDI,FONDKAPREMONT_MODE,HOUSETYPE_MODE,TOTALAREA_MODE,WALLSMATERIAL_MODE,EMERGENCYSTATE_MODE,OBS_30_CNT_SOCIAL_CIRCLE,DEF_30_CNT_SOCIAL_CIRCLE,OBS_60_CNT_SOCIAL_CIRCLE,DEF_60_CNT_SOCIAL_CIRCLE,DAYS_LAST_PHONE_CHANGE,FLAG_DOCUMENT_2,FLAG_DOCUMENT_3,FLAG_DOCUMENT_4,FLAG_DOCUMENT_5,FLAG_DOCUMENT_6,FLAG_DOCUMENT_7,FLAG_DOCUMENT_8,FLAG_DOCUMENT_9,FLAG_DOCUMENT_10,FLAG_DOCUMENT_11,FLAG_DOCUMENT_12,FLAG_DOCUMENT_13,FLAG_DOCUMENT_14,FLAG_DOCUMENT_15,FLAG_DOCUMENT_16,FLAG_DOCUMENT_17,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR,BUREAU_DAYS_CREDIT_MEAN,BUREAU_DAYS_CREDIT_SUM,BUREAU_DAYS_CREDIT_MIN,BUREAU_DAYS_CREDIT_MAX,BUREAU_CREDIT_DAY_OVERDUE_MEAN,BUREAU_CREDIT_DAY_OVERDUE_SUM,BUREAU_CREDIT_DAY_OVERDUE_MIN,BUREAU_CREDIT_DAY_OVERDUE_MAX,BUREAU_DAYS_CREDIT_ENDDATE_MEAN,BUREAU_DAYS_CREDIT_ENDDATE_SUM,BUREAU_DAYS_CREDIT_ENDDATE_MIN,BUREAU_DAYS_CREDIT_ENDDATE_MAX,BUREAU_DAYS_ENDDATE_FACT_MEAN,BUREAU_DAYS_ENDDATE_FACT_SUM,BUREAU_DAYS_ENDDATE_FACT_MIN,BUREAU_DAYS_ENDDATE_FACT_MAX,BUREAU_AMT_CREDIT_MAX_OVERDUE_MEAN,BUREAU_AMT_CREDIT_MAX_OVERDUE_SUM,BUREAU_AMT_CREDIT_MAX_OVERDUE_MIN,BUREAU_AMT_CREDIT_MAX_OVERDUE_MAX,BUREAU_CNT_CREDIT_PROLONG_MEAN,BUREAU_CNT_CREDIT_PROLONG_SUM,BUREAU_CNT_CREDIT_PROLONG_MIN,BUREAU_CNT_CREDIT_PROLONG_MAX,BUREAU_AMT_CREDIT_SUM_MEAN,BUREAU_AMT_CREDIT_SUM_SUM,BUREAU_AMT_CREDIT_SUM_MIN,BUREAU_AMT_CREDIT_SUM_MAX,BUREAU_AMT_CREDIT_SUM_DEBT_MEAN,BUREAU_AMT_CREDIT_SUM_DEBT_SUM,BUREAU_AMT_CREDIT_SUM_DEBT_MIN,BUREAU_AMT_CREDIT_SUM_DEBT_MAX,BUREAU_AMT_CREDIT_SUM_LIMIT_MEAN,BUREAU_AMT_CREDIT_SUM_LIMIT_SUM,BUREAU_AMT_CREDIT_SUM_LIMIT_MIN,BUREAU_AMT_CREDIT_SUM_LIMIT_MAX,BUREAU_AMT_CREDIT_SUM_OVERDUE_MEAN,BUREAU_AMT_CREDIT_SUM_OVERDUE_SUM,BUREAU_AMT_CREDIT_SUM_OVERDUE_MIN,BUREAU_AMT_CREDIT_SUM_OVERDUE_MAX,BUREAU_DAYS_CREDIT_UPDATE_MEAN,BUREAU_DAYS_CREDIT_UPDATE_SUM,BUREAU_DAYS_CREDIT_UPDATE_MIN,BUREAU_DAYS_CREDIT_UPDATE_MAX,BUREAU_AMT_ANNUITY_MEAN,BUREAU_AMT_ANNUITY_SUM,BUREAU_AMT_ANNUITY_MIN,BUREAU_AMT_ANNUITY_MAX,BUREAU_BB_MONTHS_COUNT_MEAN,BUREAU_BB_MONTHS_COUNT_SUM,BUREAU_BB_MONTHS_COUNT_MIN,BUREAU_BB_MONTHS_COUNT_MAX,BUREAU_BB_MONTHS_MIN_MEAN,BUREAU_BB_MONTHS_MIN_SUM,BUREAU_BB_MONTHS_MIN_MIN,BUREAU_BB_MONTHS_MIN_MAX,BUREAU_BB_MONTHS_MAX_MEAN,BUREAU_BB_MONTHS_MAX_SUM,BUREAU_BB_MONTHS_MAX_MIN,BUREAU_BB_MONTHS_MAX_MAX,BUREAU_BB_STATUS_0_MEAN,BUREAU_BB_STATUS_0_SUM,BUREAU_BB_STATUS_0_MIN,BUREAU_BB_S

## Save the feature engineered datasets

In [25]:
OUTPUT_DIR = Path("./output")

train_final.to_parquet(
    OUTPUT_DIR / "train_feature_engineered.parquet",
    index=False
)

test_final.to_parquet(
    OUTPUT_DIR / "test_feature_engineered.parquet",
    index=False
)

print(" Feature engineered datasets saved.")

 Feature engineered datasets saved.


### Feature Engineering Summary

Feature engineering was performed to create additional variables that could improve the prediction of loan default. New features were created using domain knowledge by combining existing variables into meaningful ratios, averages, totals, and time-based measures. The engineered features were validated by checking for missing values, infinite values, and summary statistics. Correlation analysis showed that the combined external credit score features had stronger relationships with the target than the original variables, indicating that the new features added useful predictive information. The final engineered datasets were then saved for model training.